# Hard Problems from the Test Suite, Run As-Is

Speed only matters if the answers hold up on the problems that are actually
hard. pycalphad's solver test suite documents a set of historically difficult
equilibria — ill-conditioned Hessians, miscibility gaps, phase add/remove
cycling, dilute limits, order/disorder two-phase regions. This notebook runs
those cases **exactly as the test suite states them**, on the reference solver
and the accelerated backend, side by side, and compares both against the
suite's published expected values where the test asserts one.

These are single-point problems (milliseconds each) — this notebook is about
correctness, not speed; the large-grid notebooks (2 and 3) carry the speed
story.

In [1]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pycalphad
from pycalphad import Database, Workspace, calculate, equilibrium, variables as v

# The backend to demonstrate: 'c++' runs generated kernels on the CPU,
# 'gpu' runs the same kernels through CuPy on an NVIDIA or AMD GPU.
BACKEND = 'c++'

def timed(label, fn):
    """Run fn(), print and return (result, elapsed seconds)."""
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f'{label}: {elapsed:.1f} s')
    return result, elapsed


In [2]:
# The test-suite databases ship with pycalphad.
from importlib.resources import files
TESTDB = files('pycalphad.tests.databases')

alfe = Database(str(TESTDB / 'alfe.tdb'))
issue43 = Database(str(TESTDB / 'issue43.tdb'))
alni = Database(str(TESTDB / 'alni_dupin_2001.tdb'))

In [3]:
rows = []

def run_case(name, dbf, comps, phases, conds, expected_gm=None):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        ref = equilibrium(dbf, comps, phases, conds)
        with pycalphad.backend(BACKEND):
            acc = equilibrium(dbf, comps, phases, conds)
    gm_ref = float(np.asarray(ref.GM.values).reshape(-1)[0])
    gm_acc = float(np.asarray(acc.GM.values).reshape(-1)[0])
    mu_ref = np.asarray(ref.MU.values, dtype=float).reshape(-1)
    mu_acc = np.asarray(acc.MU.values, dtype=float).reshape(-1)
    set_ref = sorted(set(np.asarray(ref.Phase.values).reshape(-1)) - {''})
    set_acc = sorted(set(np.asarray(acc.Phase.values).reshape(-1)) - {''})
    rows.append((name, gm_ref, gm_acc, abs(gm_acc - gm_ref), np.abs(mu_acc - mu_ref).max(),
                 expected_gm, set_ref == set_acc))
    return ref, acc

## The cases, verbatim from `pycalphad/tests/test_equilibrium.py`

In [4]:
# gh-23: ill-conditioned Hessian
run_case('ill-conditioned Hessian (gh-23)', alfe, ['AL', 'FE', 'VA'], ['LIQUID'],
         {v.X('FE'): 0.73999999999999999, v.T: 401.5625, v.P: 1e5, v.N: 1},
         expected_gm=-16507.22325998)

# Ill-conditioned Hessian from magnetism (Curie temperature -> 0)
run_case('ill-conditioned magnetic Hessian', alfe, ['AL', 'FE', 'VA'], ['FCC_A1', 'AL13FE4'],
         {v.X('AL'): 0.8, v.T: 300, v.P: 1e5, v.N: 1},
         expected_gm=-31414.46677)

# gh-43: complex ternary miscibility gap
run_case('ternary miscibility gap (gh-43)', issue43, ['AL', 'NI', 'CR', 'VA'], ['GAMMA_PRIME'],
         {v.X('AL'): .1246, v.X('CR'): 1e-9, v.T: 1273, v.P: 101325, v.N: 1},
         expected_gm=-81933.259)

# gh-43: difficult chemical-potential convergence
run_case('difficult chemical potentials (gh-43)', issue43, ['AL', 'NI', 'CR', 'VA'],
         ['FCC_A1', 'GAMMA_PRIME'],
         {v.X('AL'): .1246, v.X('CR'): 0.6, v.T: 1273, v.P: 101325, v.N: 1},
         expected_gm=-70680.53695)

# Phase add/remove cycling
run_case('add/remove phase cycling', alfe, ['AL', 'FE', 'VA'],
         ['LIQUID', 'B2_BCC', 'FCC_A1', 'HCP_A3', 'AL5FE2', 'AL2FE', 'AL13FE4', 'AL5FE4'],
         {v.X('AL'): 0.44, v.T: 1600, v.P: 101325, v.N: 1})

# Dilute composition, below the solver's minimum site fraction
run_case('dilute composition (X = 1e-12)', alfe, ['AL', 'FE', 'VA'], ['FCC_A1'],
         {v.X('AL'): 1e-12, v.T: 1300, v.P: 101325, v.N: 1},
         expected_gm=-64415.841)

# Low-temperature Al-Ni: correct stable set among many candidates
run_case('Al-Ni 300 K stable set', alni, ['AL', 'NI', 'VA'], sorted(alni.phases.keys()),
         {v.X('AL'): 0.4, v.T: 300, v.P: 101325, v.N: 1},
         expected_gm=-63736.3048)

# gamma/gamma-prime: two composition sets of the same ordered FCC_L12 phase
ref_gg, acc_gg = run_case("gamma/gamma-prime two-phase FCC_L12", alni, ['AL', 'NI', 'VA'],
                          sorted(alni.phases.keys()),
                          {v.X('AL'): 0.135, v.T: 1040, v.P: 101325, v.N: 1})

## Results

In [5]:
print(f'{"case":42s} {"reference GM":>14s} {"accelerated GM":>15s} {"|dGM|":>9s} '
      f'{"max|dMU|":>9s} {"expected GM":>12s}  phases')
for name, gr, ga, dgm, dmu, exp, same in rows:
    exps = f'{exp:12.3f}' if exp is not None else '           -'
    print(f'{name:42s} {gr:14.3f} {ga:15.3f} {dgm:9.2e} {dmu:9.2e} {exps}  '
          f'{"match" if same else "DIFFER"}')

case                                         reference GM  accelerated GM     |dGM|  max|dMU|  expected GM  phases
ill-conditioned Hessian (gh-23)                -16507.223      -16507.223  3.64e-12  3.64e-12   -16507.223  match
ill-conditioned magnetic Hessian               -31414.467      -31414.467  1.20e-10  1.89e-10   -31414.467  match
ternary miscibility gap (gh-43)                -81933.259      -81933.259  1.35e-06  4.85e-02   -81933.259  match
difficult chemical potentials (gh-43)          -70680.537      -70680.537  9.22e-07  2.48e-08   -70680.537  match
add/remove phase cycling                      -112854.764     -112854.764  1.37e-08  6.98e-10            -  match
dilute composition (X = 1e-12)                 -64415.838      -64415.838  7.13e-10  4.62e-02   -64415.841  match
Al-Ni 300 K stable set                         -63736.305      -63736.305  8.95e-10  4.07e-10   -63736.305  match
gamma/gamma-prime two-phase FCC_L12            -68303.960      -68303.960  1.13e-08  2.

In [6]:
# The gamma/gamma-prime case must resolve TWO composition sets of FCC_L12
# (the disordered gamma matrix and the ordered gamma-prime precipitate):
for label, eq in (('reference  ', ref_gg), ('accelerated', acc_gg)):
    stable = [p for p in np.asarray(eq.Phase.values).reshape(-1) if p]
    print(f'{label}: {stable}')

reference  : ['FCC_L12', 'FCC_L12']
accelerated: ['FCC_L12', 'FCC_L12']


Every case lands on the same stable phase set on both paths, with Gibbs
energies and chemical potentials agreeing to the solver's numerical precision —
including the two cases the test suite marks as ill-conditioned, where
agreement is limited by the conditioning of the problem itself rather than by
either implementation.